# Setup

In [3]:
import os, sys

SUMO_HOME  = '/Library/Frameworks/EclipseSUMO.framework/Versions/1.22.0/EclipseSUMO/share/sumo'
PROJ_PATH  = '/Library/Frameworks/EclipseSUMO.framework/Versions/1.22.0/EclipseSUMO/framework/EclipseSUMO.framework/Versions/1.22.0/EclipseSUMO/share/proj'

os.environ['SUMO_HOME']  = SUMO_HOME
os.environ['PROJ_LIB']   = PROJ_PATH
os.environ['PROJ_DATA']  = PROJ_PATH

sys.path.append(os.path.join(SUMO_HOME, 'tools'))

# Verify all three
checks = {
    'SUMO_HOME':  os.path.exists(SUMO_HOME),
    'proj.db':    os.path.exists(os.path.join(PROJ_PATH, 'proj.db')),
    'gtfs2pt.py': os.path.exists(f'{SUMO_HOME}/tools/import/gtfs/gtfs2pt.py'),
}
for k, v in checks.items():
    print(f"{k:15s}: {'✅' if v else '❌ NOT FOUND'}")

SUMO_HOME      : ✅
proj.db        : ✅
gtfs2pt.py     : ✅


In [5]:
cd /Users/ziliqu/Dev/SDOT_Worldcup/Sumo_Test/2

/Users/ziliqu/Dev/SDOT_Worldcup/Sumo_Test/2


# OSM Map Extraction

-extraction from Geofrabil for Full Washington Link Network

In [24]:
import requests

# Download Washington state extract from Geofabrik
url = "https://download.geofabrik.de/north-america/us/washington-latest.osm.pbf"
out = "washington.osm.pbf"

print("Downloading Washington state from Geofabrik (~100MB)...")
with requests.get(url, stream=True) as r:
    total = int(r.headers.get('content-length', 0))
    downloaded = 0
    with open(out, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f"\r  {pct:.1f}% ({downloaded/1024/1024:.1f} MB)", end='')

print(f"\n✅ Done — saved to {out}")

  100.0% (339.4 MB)
✅ Done — saved to washington.osm.pbf


In [52]:
!osmium extract \
  -b -122.4500,47.2500,-122.1000,47.8500 \
  washington.osm.pbf \
  -o seattle_link.osm.pbf \
  --strategy complete_ways

No extract specified in config file or on the command line.


In [56]:
!osmium extract \
  -b -124.8,45.5,-116.9,49.1 \
  washington.osm.pbf \
  -o washington_full.osm.pbf \
  --strategy complete_ways

[======================================================================] 100% 


In [36]:
!osmium cat seattle_link.osm.pbf -o seattle_link.osm.xml

[======================================================================] 100% 


In [58]:
!osmium cat washington_full.osm.pbf -o seattle_link_full.osm.xml

[======================================================================] 100% 


# Network Preparation

In [ ]:
!netconvert \
  --osm-files seattle_link.osm.xml \
  -o seattle_lightrail_3.net.xml \
  --type-files $SUMO_HOME/data/typemap/osmNetconvert.typ.xml,$SUMO_HOME/data/typemap/osmNetconvertRailUsage.typ.xml \
  --keep-edges.by-type railway.light_rail,railway.subway \
  --proj.utm true \
  --geometry.remove \
  --junctions.join \
  --output.street-names \
  --ptstop-output seattle_rail_stops.add.xml \
  --ptline-output seattle_rail_ptlines.add.xml \
  --osm.stop-output.length 30

# GTFS

### Filtering to major Seattle GTFS Bus Lines

In [69]:
import pandas as pd
import zipfile
import os

# ==============================
# CONFIG (EDIT THESE)
# ==============================

INPUT_ZIP = "google_transit.zip"
OUTPUT_ZIP = "google_transit_downtown.zip"

# Select important downtown routes 
KEEP_ROUTES = [
    "1", "2", "3", "4", "7", "8", "40",
    "101", "120", "C Line", "D Line", "E Line", "H Line"
]

# Downtown Seattle bounding box
LAT_MIN, LAT_MAX = 47.58, 47.65
LON_MIN, LON_MAX = -122.37, -122.30

# ==============================
# LOAD GTFS FILES
# ==============================

def load_gtfs(zip_path):
    data = {}
    with zipfile.ZipFile(zip_path, 'r') as z:
        for file in z.namelist():
            if file.endswith(".txt"):
                data[file] = pd.read_csv(z.open(file))
    return data

gtfs = load_gtfs(INPUT_ZIP)

routes = gtfs["routes.txt"]
trips = gtfs["trips.txt"]
stop_times = gtfs["stop_times.txt"]
stops = gtfs["stops.txt"]

# ==============================
# STEP 1: FILTER ROUTES
# ==============================

filtered_routes = routes[routes["route_short_name"].isin(KEEP_ROUTES)]

print(f"Kept routes: {len(filtered_routes)}")

# ==============================
# STEP 2: FILTER TRIPS
# ==============================

filtered_trips = trips[trips["route_id"].isin(filtered_routes["route_id"])]

print(f"Kept trips: {len(filtered_trips)}")

# ==============================
# STEP 3: FILTER STOP TIMES
# ==============================

filtered_stop_times = stop_times[
    stop_times["trip_id"].isin(filtered_trips["trip_id"])
]

# ==============================
# STEP 4: FILTER STOPS (GEOGRAPHIC)
# ==============================

filtered_stops = stops[
    (stops["stop_lat"] >= LAT_MIN) &
    (stops["stop_lat"] <= LAT_MAX) &
    (stops["stop_lon"] >= LON_MIN) &
    (stops["stop_lon"] <= LON_MAX)
]

print(f"Kept stops in downtown: {len(filtered_stops)}")

# Keep only stop_times that use those stops
filtered_stop_times = filtered_stop_times[
    filtered_stop_times["stop_id"].isin(filtered_stops["stop_id"])
]

# ==============================
# STEP 5: CLEAN TRIPS (remove empty)
# ==============================

valid_trip_ids = filtered_stop_times["trip_id"].unique()
filtered_trips = filtered_trips[filtered_trips["trip_id"].isin(valid_trip_ids)]

# ==============================
# STEP 6: WRITE NEW GTFS ZIP
# ==============================

with zipfile.ZipFile(OUTPUT_ZIP, 'w') as z:
    def write(df, name):
        z.writestr(name, df.to_csv(index=False))

    write(filtered_routes, "routes.txt")
    write(filtered_trips, "trips.txt")
    write(filtered_stop_times, "stop_times.txt")
    write(filtered_stops, "stops.txt")

    # Copy unchanged files if they exist
    for fname in ["calendar.txt", "calendar_dates.txt", "agency.txt"]:
        if fname in gtfs:
            write(gtfs[fname], fname)

print(f"✅ Filtered GTFS saved to: {OUTPUT_ZIP}")


Kept routes: 12
Kept trips: 5413
Kept stops in downtown: 649
✅ Filtered GTFS saved to: google_transit_downtown.zip


In [8]:
import pandas as pd
import zipfile
import os

# ==============================
# CONFIG (EDIT THESE)
# ==============================

INPUT_ZIP = "rail_gtfs.zip"
OUTPUT_ZIP = "rail_gtfs_fil.zip"

# Select important downtown routes 
KEEP_ROUTES = [
    "1 Line", "2 Line"
]

# Downtown Seattle bounding box
LAT_MIN, LAT_MAX = 47.31, 47.84
LON_MIN, LON_MAX = -122.43, -122.09

# ==============================
# LOAD GTFS FILES
# ==============================

def load_gtfs(zip_path):
    data = {}
    with zipfile.ZipFile(zip_path, 'r') as z:
        for file in z.namelist():
            if file.endswith(".txt"):
                data[file] = pd.read_csv(z.open(file))
    return data

gtfs = load_gtfs(INPUT_ZIP)

routes = gtfs["routes.txt"]
trips = gtfs["trips.txt"]
stop_times = gtfs["stop_times.txt"]
stops = gtfs["stops.txt"]

# ==============================
# STEP 1: FILTER ROUTES
# ==============================

filtered_routes = routes[routes["route_short_name"].isin(KEEP_ROUTES)]

print(f"Kept routes: {len(filtered_routes)}")

# ==============================
# STEP 2: FILTER TRIPS
# ==============================

filtered_trips = trips[trips["route_id"].isin(filtered_routes["route_id"])]

print(f"Kept trips: {len(filtered_trips)}")

# ==============================
# STEP 3: FILTER STOP TIMES
# ==============================

filtered_stop_times = stop_times[
    stop_times["trip_id"].isin(filtered_trips["trip_id"])
]

# ==============================
# STEP 4: FILTER STOPS (GEOGRAPHIC)
# ==============================

filtered_stops = stops[
    (stops["stop_lat"] >= LAT_MIN) &
    (stops["stop_lat"] <= LAT_MAX) &
    (stops["stop_lon"] >= LON_MIN) &
    (stops["stop_lon"] <= LON_MAX)
]

print(f"Kept stops in downtown: {len(filtered_stops)}")

# Keep only stop_times that use those stops
filtered_stop_times = filtered_stop_times[
    filtered_stop_times["stop_id"].isin(filtered_stops["stop_id"])
]

# ==============================
# STEP 5: CLEAN TRIPS (remove empty)
# ==============================

valid_trip_ids = filtered_stop_times["trip_id"].unique()
filtered_trips = filtered_trips[filtered_trips["trip_id"].isin(valid_trip_ids)]

# ==============================
# STEP 6: WRITE NEW GTFS ZIP
# ==============================

with zipfile.ZipFile(OUTPUT_ZIP, 'w') as z:
    def write(df, name):
        z.writestr(name, df.to_csv(index=False))

    write(filtered_routes, "routes.txt")
    write(filtered_trips, "trips.txt")
    write(filtered_stop_times, "stop_times.txt")
    write(filtered_stops, "stops.txt")

    # Copy unchanged files if they exist
    for fname in ["calendar.txt", "calendar_dates.txt", "agency.txt"]:
        if fname in gtfs:
            write(gtfs[fname], fname)

print(f"✅ Filtered GTFS saved to: {OUTPUT_ZIP}")


Kept routes: 2
Kept trips: 16865
Kept stops in downtown: 311
✅ Filtered GTFS saved to: rail_gtfs_fil.zip


### Converging new Lightrail Segments

In [ ]:
!netconvert -s soheil_seattle.net.xml --remove-edges.by-vclass rail,rail_urban,rail_fast,subway,rail_electric -o soheil_seattle_veh.net.xml

In [ ]:
!netconvert -s seattle_lightrail_2.net.xml,soheil_seattle_veh.net.xml -o merged.net.xml 

#### Adding Full Lightrail Network

In [ ]:
!netconvert -s seattle_lightrail_trimmed.net.xml,soheil_seattle_original.net.xml -o merged.net.xml 

In [ ]:
!netconvert -s seattle_lightrail_trimmed1.net.xml,merged.net.xml  -o merged_final.net.xml 

### Adding GTFS to Sumo Network

In [10]:
!python $SUMO_HOME/tools/import/gtfs/gtfs2pt.py \
-n merged.net.xml \
--gtfs rail_gtfs_fil.zip \
--date 20260329 \
--modes=tram \
--repair \
--verbose

Loading net
function import_gtfs called at Mon, 15 Jun 2026 14:04:24 +0000
Loading GTFS data "rail_gtfs_fil.zip"
function import_gtfs finished after 0.680602 seconds
Writing fcd file "fcd/gtfs/tram.fcd.xml"
Reusing old resources/gtfs/numerical.net.xml
Reusing old resources/gtfs/tram.net.xml
Warning! No mapping library found, falling back to tracemapper.
mapping tram
mapping trace with 15 points
mapping trace with 26 points
mapping trace with 22 points
mapping trace with 26 points
mapping trace with 4 points
mapping trace with 8 points
mapping trace with 11 points
mapping trace with 22 points
mapping trace with 11 points
mapping trace with 26 points
mapping trace with 26 points
mapping trace with 1 points
mapping trace with 4 points
mapping trace with 6 points
mapping trace with 10 points
mapping trace with 19 points
mapping trace with 24 points
mapping trace with 23 points
mapping trace with 2 points
mapping trace with 6 points
mapping trace with 5 points
mapping trace with 4 points
ma

In [13]:
!python $SUMO_HOME/tools/import/gtfs/gtfs2pt.py \
-n merged.net.xml \
--gtfs google_transit_downtown.zip \
--date 20260617 \
--modes=bus \
--repair \
--verbose

usage: gtfs2pt.py [-h] [-c FILE] [-C FILE] [--save-template FILE] [-r REGION]
                  --gtfs GTFS --date DATE [--fcd FCD] [--gpsdat GPSDAT]
                  [--modes MODES] [--vtype-output VTYPE_OUTPUT] [-v]
                  [-b BEGIN] [-e END] [--bbox BBOX] -n NETWORK
                  [--route-output ROUTE_OUTPUT]
                  [--additional-output ADDITIONAL_OUTPUT]
                  [--duration DURATION] [--bus-stop-length BUS_STOP_LENGTH]
                  [--train-stop-length TRAIN_STOP_LENGTH]
                  [--tram-stop-length TRAM_STOP_LENGTH] [--center-stops]
                  [--skip-access] [--sort] [--stops STOPS] [-H]
                  [--network-split NETWORK_SPLIT] [--network-split-vclass]
                  [--warn-unmapped] [--mapperlib MAPPERLIB]
                  [--map-output MAP_OUTPUT]
                  [--map-output-config MAP_OUTPUT_CONFIG]
                  [--map-input-config MAP_INPUT_CONFIG]
                  [--map-parameter MAP_PARAMETER

In [ ]:

with open("tram.add.xml", "w") as f:
    f.write("""<additional>
    <vType id="tram"
           vClass="rail"
           accel="1.2"
           decel="1.5"
           sigma="0.3"
           length="40"
           maxSpeed="25"
           guiShape="rail"/>
</additional>
""")


In [6]:
with open("bus.add.xml", "w") as f:
    f.write("""<additional>
    <vType id="bus"
           vClass="bus"
           accel="1.0"
           decel="4.5"
           sigma="0.5"
           length="12"
           maxSpeed="16.7"
           guiShape="bus"/>
</additional>
""")


In [12]:
import re

with open("gtfs_pt_stops.add.xml") as f:
    content = f.read()

# replace trainStop with busStop
content = content.replace("trainStop", "busStop")

with open("gtfs_pt_stops_fixed.add.xml", "w") as f:
    f.write(content)


In [ ]:
!sumo-gui \
-n merged.net.xml \
-a tram.add.xml,gtfs_pt_stops_fixed.add.xml,gtfs_pt_vehicles.add.xml


In [ ]:
!sumo-gui \
-n merged.net.xml \
-a bus.add.xml,gtfs_pt_stops.add.xml,gtfs_pt_vehicles.add.xml